# Export the completed SFT model to safetensors (CPU)

Choose **CPU**, enable **Internet**, and attach the saved TPU posttraining
output dataset. Keep its `training/` folder intact, including `tokenizer.json`,
`corpus_manifest.json`, and `sft/checkpoints/`. No corpus shards or pretrain
dataset are required. The finished run's final step is
`step-00085663-af2e759ac4dc` under `posttrain-20261001-183833/training/sft/checkpoints/`.

This exports only the completed SFT model parameters, including DSpark weights,
with their original BF16/FP32 dtypes. It reads the effective SFT recipe so the
32K context and compressed-layer YaRN factor 4 survive export. Optimizer files
are never loaded. Every model tensor is checksum-checked and compared bit for
bit after safetensors export. The bundled tokenizer must match the training hash.

The output uses the project's portable layout for its custom CPU runtime.
Save the original training checkpoint too if you want to resume training.


In [ ]:
SOURCE_REF = 'codex/midtrain8k-sft16k'
CHECKPOINT = ''  # blank: locate exactly one SFT checkpoint set under /kaggle/input
OUTPUT = '/kaggle/working/nano-dsv41f-sft32k-safetensors'  # use a fresh directory


In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

if any(name in sys.modules for name in ('jax', 'nano_dsv41f')):
    raise RuntimeError('Restart the session before updating the source and dependencies')
os.environ['JAX_PLATFORMS'] = 'cpu'
REPO = Path('/kaggle/working/nano-dsv4.1f')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/xiayicheng3-code/nano-dsv4.1f.git', str(REPO)], check=True)
if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain',
                            '--untracked-files=no'], text=True).strip():
    raise RuntimeError('Preserve tracked edits before updating this checkout')
subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', SOURCE_REF], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
if importlib.util.find_spec('torch') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                    '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(REPO / 'requirements-posttrain-inference.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), '--no-deps'], check=True)
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(REPO / 'scripts'))
print('Source commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import json
from export_posttrain_checkpoint import resolve_checkpoint
from pretrain_checkpoint import read_metadata

selected = resolve_checkpoint(CHECKPOINT or '/kaggle/input')
_, checkpoint_manifest = read_metadata(selected)
state = checkpoint_manifest['metadata']
print('Selected checkpoint:', selected)
print('Stage:', state.get('stage'), 'complete:', state.get('stage_complete'))
print('Global step:', state['completed_steps'])
print('SFT context:', state['recipe']['train']['seq_len'])
print('RoPE:', json.dumps(state['recipe']['model']['attention']['rope'], indent=2))
command = [sys.executable, '-u', str(REPO / 'scripts/export_posttrain_checkpoint.py'),
           '--checkpoint', str(selected), '--output', OUTPUT]
subprocess.run(command, check=True)


In [ ]:
from nano_dsv41f.portable_bundle import verify_portable_bundle
from IPython.display import FileLink, display
report = verify_portable_bundle(OUTPUT)
print('Verified tensors:', report['tensor_count'], 'parameters:', report['parameter_count'])
print('Context length:', report['max_position_embeddings'])
for path in sorted(Path(OUTPUT).iterdir()):
    display(FileLink(str(path)))


## Save and use

Save this notebook version with its outputs, then save the entire
`nano-dsv41f-sft32k-safetensors` folder as a Kaggle dataset. Keep all files
together, especially `model.safetensors`, `config.json`, `tokenizer.json`,
`training_recipe.json`, and `export_manifest.json`. Attach this exported
dataset to `nano_dsv41f_sft_inference_cpu.ipynb`. GitHub supplies the runtime
code; the weights remain in your chosen dataset. No upload is automatic.
